# Benchmark 4 audio trên Qwen base, ViMD-small và ViMD-full
Chạy tuần tự từng model để hạn chế VRAM. Ba audio đầu và reference được lấy
trực tiếp từ split test ViMD; super-hard được lấy từ package external instance.
Kết quả gồm prediction từng mẫu, CER/WER từng mẫu và metric gộp theo model.



In [ ]:
from dataclasses import replace
from datetime import datetime
from pathlib import Path
import gc
import json

import pandas as pd
import torch

try:
    from _bootstrap import initialize_project
except ImportError:
    from nb._bootstrap import initialize_project

ROOT = initialize_project()
from mdv_asr.config.loader import ConfigLoader
from mdv_asr.config.types import DatasetConfig
from mdv_asr.data.audio import AudioReader
from mdv_asr.data.sources import DatasetSourceFactory
from mdv_asr.evaluation.metrics import ErrorCounter, MetricAggregator
from mdv_asr.evaluation.records import PredictionRecord
from mdv_asr.models.factory import ASRBackendFactory



## Cấu hình
Trên Kaggle, sửa SMALL_ADAPTER_PATH và FULL_ADAPTER_PATH thành thư mục model
đã Add Input (thư mục phải có adapter_config.json và adapter_model.safetensors).
SUPERHARD_ROOT trỏ đến dataset/package đã giải nén, có sample_metadata.json.



In [ ]:
CONFIG_PATH = ROOT / "configs/benchmark/vimd_three_models.json"
SMALL_ADAPTER_PATH = ROOT / "models/qwen-vimd-small"
FULL_ADAPTER_PATH = ROOT / "models/qwen-vimd-full"
SUPERHARD_ROOT = ROOT / "outputs/super_hard_instance"
VIMD_FILENAMES = ["30_0309.wav", "59_0313.wav", "37_0338.wav"]
SUPERHARD_SAMPLE_ID = "test_super_hard_000000"
OUTPUT_ROOT = Path("/kaggle/working/qwen_4_audio_benchmark") if Path("/kaggle/working").is_dir() \
    else ROOT / "outputs/benchmark_models/qwen_4_audio_3_models"
RUN_DIR = OUTPUT_ROOT / datetime.now().strftime("run_%Y%m%d_%H%M%S")
RUN_DIR.mkdir(parents=True, exist_ok=False)

config = ConfigLoader().load(CONFIG_PATH)
if config.benchmark is None:
    raise ValueError(f"Thiếu benchmark config trong {CONFIG_PATH}")

adapter_paths = {
    "qwen_small_vimd": Path(SMALL_ADAPTER_PATH).expanduser().resolve(),
    "qwen_full_vimd": Path(FULL_ADAPTER_PATH).expanduser().resolve(),
}
for model_key, path in adapter_paths.items():
    for required in ("adapter_config.json", "adapter_model.safetensors"):
        if not (path / required).is_file():
            raise FileNotFoundError(f"{model_key}: thiếu {required} trong {path}; chỉnh adapter path ở cell config")

model_specs = []
for spec in config.benchmark.models:
    if spec.key == "qwen_small_vimd":
        spec = replace(spec, adapter_path=str(adapter_paths[spec.key]))
    elif spec.key == "qwen_full_vimd":
        spec = replace(spec, adapter_path=str(adapter_paths[spec.key]))
    elif spec.key != "qwen_base":
        raise ValueError(f"Model không nằm trong benchmark 3-model đã chốt: {spec.key}")
    model_specs.append(spec)
if {spec.key for spec in model_specs} != {"qwen_base", "qwen_small_vimd", "qwen_full_vimd"}:
    raise ValueError(f"Cần đủ qwen_base/small/full; hiện có {[spec.key for spec in model_specs]}")

print("Models:")
for spec in model_specs:
    print(f"- {spec.key}: {spec.display_name or spec.model_id} | adapter={spec.adapter_path}")
print("Super-hard package:", SUPERHARD_ROOT)
print("Output:", RUN_DIR)



## Lấy đúng bốn audio và reference
Không phụ thuộc vào bảng/filter vùng miền trong notebook khác. Dataset ViMD và
package super-hard được mở bằng hai DatasetSource riêng, nhưng dùng chung AudioReader.



In [ ]:
dataset = DatasetSourceFactory.create(config.dataset)
superhard_source = None
reader = AudioReader(target_sample_rate=16000)
cases = []
try:
    vimd_index = dataset.build_index()
    for filename in VIMD_FILENAMES:
        record = vimd_index.find_by_filename(filename, split="test")
        audio = reader.read(record, dataset)
        reader.validate_duration(record, audio)
        cases.append({"record": record, "audio": audio, "source": "ViMD"})

    package_config = DatasetConfig(
        kind="package", root=str(SUPERHARD_ROOT), dataset_id="super_hard_audio",
        allow_download=False,
    )
    superhard_source = DatasetSourceFactory.create(package_config)
    superhard_index = superhard_source.build_index()
    superhard_record = superhard_index.get(SUPERHARD_SAMPLE_ID, dataset_id="super_hard_audio")
    superhard_audio = reader.read(superhard_record, superhard_source)
    reader.validate_duration(superhard_record, superhard_audio)
    cases.append({"record": superhard_record, "audio": superhard_audio, "source": "super-hard"})
except Exception:
    dataset.close()
    if superhard_source is not None:
        superhard_source.close()
    raise

if len(cases) != 4 or len({case["record"].key for case in cases}) != 4:
    raise ValueError("Expected exactly four unique test instances")

for case in cases:
    record, audio = case["record"], case["audio"]
    print(f"{case['source']:10s} | {record.sample_id:28s} | {record.filename:20s} "
          f"| {audio.duration_seconds:6.2f}s | {record.province_name or 'unknown'}")
    print("REF:", record.text)



## Chạy lần lượt ba model
Mỗi model được load một lần, chạy trên cùng bốn waveform rồi giải phóng trước
khi nạp model tiếp theo. Các lỗi inference được ghi thành dòng trong CSV.



In [ ]:
counter = ErrorCounter()
prediction_rows = []
for spec in model_specs:
    backend = ASRBackendFactory.create(spec)
    backend.decode_config = config.decode
    print(f"\nLoading {spec.key} ...", flush=True)
    try:
        backend.load()
        for case in cases:
            record, audio = case["record"], case["audio"]
            try:
                result = backend.transcribe(audio, config.decode)
                counts = counter.count(record.text, result.text)
                prediction_rows.append({
                    "dataset_id": record.dataset_id, "sample_id": record.sample_id,
                    "source": case["source"], "filename": record.filename,
                    "model_key": spec.key, "model_name": spec.display_name or spec.key,
                    "reference": record.text, "prediction": result.text,
                    "wer_errors": counts.word_errors, "reference_words": counts.reference_words,
                    "WER": counts.wer, "cer_errors": counts.char_errors,
                    "reference_chars": counts.reference_chars, "CER": counts.cer,
                    "inference_seconds": result.elapsed_seconds, "error": "",
                })
                print(f"[{spec.key} | {record.filename}] CER={counts.cer:.4f} WER={counts.wer:.4f}")
                print("HYP:", result.text)
            except Exception as exc:
                prediction_rows.append({
                    "dataset_id": record.dataset_id, "sample_id": record.sample_id,
                    "source": case["source"], "filename": record.filename,
                    "model_key": spec.key, "model_name": spec.display_name or spec.key,
                    "reference": record.text, "prediction": "", "wer_errors": None,
                    "reference_words": None, "WER": None, "cer_errors": None,
                    "reference_chars": None, "CER": None, "inference_seconds": None,
                    "error": repr(exc),
                })
                print(f"INFERENCE ERROR [{spec.key} | {record.filename}]: {exc}", flush=True)
    finally:
        backend.close()
        del backend
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()



## Metric tổng hợp và lưu kết quả
CER/WER overall là tổng lỗi chia tổng đơn vị tham chiếu của bốn mẫu, không phải
trung bình đơn giản của bốn tỷ lệ. Mỗi model cần có bốn dự đoán thành công mới
được đưa vào so sánh paired.



In [ ]:
predictions = pd.DataFrame(prediction_rows)
predictions_path = RUN_DIR / "predictions.csv"
predictions.to_csv(predictions_path, index=False, encoding="utf-8-sig")

successful = [row for row in (
    PredictionRecord(
        dataset_id=str(item.dataset_id), sample_id=str(item.sample_id),
        model_key=str(item.model_key), reference=str(item.reference),
        prediction=str(item.prediction), split="test", filename=str(item.filename),
        inference_seconds=float(item.inference_seconds), error=str(item.error or ""),
        metadata={"source": str(item.source)},
    ) for item in predictions.itertuples(index=False)
) if not row.error]

aggregator = MetricAggregator(counter)
overall_rows = []
for spec in model_specs:
    model_rows = [row for row in successful if row.model_key == spec.key]
    if model_rows:
        overall_rows.append({"model_key": spec.key, "model_name": spec.display_name or spec.key,
                             **aggregator.overall(model_rows)})
overall = pd.DataFrame(overall_rows)
overall_path = RUN_DIR / "metrics_overall.csv"
overall.to_csv(overall_path, index=False, encoding="utf-8-sig")

expected = {case["record"].key for case in cases}
for spec in model_specs:
    completed = {(row.dataset_id, row.sample_id) for row in successful if row.model_key == spec.key}
    print(f"{spec.key}: {len(completed)}/{len(expected)} successful")
if len(successful) == len(model_specs) * 4:
    comparison = aggregator.compare_models(successful, baseline_key="qwen_base")
    comparison_path = RUN_DIR / "metrics_comparison.json"
    comparison_path.write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding="utf-8")
    print("\nPaired comparison (lower is better):")
    display(pd.DataFrame.from_dict(comparison, orient="index").rename_axis("model_key").reset_index())
else:
    print("\nComparison skipped: ít nhất một model có inference error hoặc thiếu sample.")

display(predictions[["source", "filename", "model_key", "CER", "WER", "inference_seconds", "error"]])
display(overall)
print("Saved:", predictions_path, overall_path)



In [ ]:
dataset.close()
if superhard_source is not None:
    superhard_source.close()
